# NOTEBOOK 99 — FINAL DEMO
### THIS IS THE FILE YOU SHOW TO FACULTY
Person 3 — Upload any video → Full multimodal analysis → Premium Visual Report

In [1]:
!pip install torch torchvision librosa soundfile opencv-python-headless tqdm scikit-learn matplotlib scipy Pillow -q

In [2]:
# Cell 2 — Mount Drive + All Imports
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, random, cv2, numpy as np, subprocess
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

BASE_DIR  = "/content/drive/MyDrive/Colab Notebooks/deepfake-project"
MODEL_DIR = os.path.join(BASE_DIR, "models")
OUT_DIR   = os.path.join(BASE_DIR, "outputs")
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(os.path.join(OUT_DIR, "demo"), exist_ok=True)
print(f"Device: {DEVICE}")

Mounted at /content/drive
Device: cpu


In [3]:
# Cell 3 — All Model Definitions
class VisualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        backbone         = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.features    = backbone.features
        self.pool        = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(1792,512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512,1))
        self.gradients = None; self.activations = None
        self.features[-1].register_forward_hook(lambda m,i,o: setattr(self,'activations',o))
        self.features[-1].register_backward_hook(lambda m,i,o: setattr(self,'gradients',o[0]))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)).flatten(1)).squeeze(1)

class TemporalNet(nn.Module):
    def __init__(self, input_dim=1792, hidden=256):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden*2, 128, batch_first=True, bidirectional=True)
        self.attn  = nn.Linear(256, 1)
        self.fc    = nn.Sequential(nn.Dropout(0.5), nn.Linear(256,64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, x):
        out,_ = self.lstm1(x); out,_ = self.lstm2(out)
        return self.fc((torch.softmax(self.attn(out),dim=1)*out).sum(1)).squeeze(1)
    def frame_scores(self, x):
        with torch.no_grad():
            out,_ = self.lstm1(x); out,_ = self.lstm2(out)
            return torch.sigmoid(self.fc(out).squeeze(-1))

class AudioCRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2))
        self.gru = nn.GRU(128*16, 128, batch_first=True, bidirectional=True)
        self.fc  = nn.Sequential(nn.Dropout(0.5), nn.Linear(256,64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, x):
        f = self.cnn(x); B,C,H,W = f.shape
        f = f.permute(0,3,1,2).reshape(B,W,C*H)
        out,_ = self.gru(f)
        return self.fc(out[:,-1,:]).squeeze(1)

class AudioEncoder(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(80,256,3,padding=1),nn.BatchNorm1d(256),nn.ReLU(),
            nn.Conv1d(256,256,3,padding=1),nn.BatchNorm1d(256),nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),nn.Flatten(),nn.Linear(256,emb_dim))
    def forward(self, x): return F.normalize(self.net(x.squeeze(1)), dim=1)

class VideoEncoder(nn.Module):
    def __init__(self, n_frames=5, emb_dim=128):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(3,32,(3,5,5),stride=(1,2,2),padding=(1,2,2)),
            nn.BatchNorm3d(32),nn.ReLU())
        self.conv2d = nn.Sequential(
            nn.Conv2d(32*n_frames,128,3,padding=1),nn.ReLU(),
            nn.Conv2d(128,256,3,padding=1),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(256,emb_dim))
    def forward(self, x):
        x = x.permute(0,2,1,3,4); f = self.conv3d(x); B,C,T,H,W = f.shape
        return F.normalize(self.conv2d(f.reshape(B,C*T,H,W)), dim=1)

class SyncNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_enc = AudioEncoder(); self.video_enc = VideoEncoder()
    def forward(self, a, v):
        return F.cosine_similarity(self.audio_enc(a), self.video_enc(v))

class FreqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Dropout(0.5),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self, x): return self.net(x).squeeze(1)

class FusionNet(nn.Module):
    MODALITIES = ["Visual","Temporal","Audio","Lip-Sync","Frequency"]
    def __init__(self):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(5,5), nn.Softmax(dim=1))
        self.fc = nn.Sequential(
            nn.Linear(10,64),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self, x):
        w = self.attention(x)
        return self.fc(torch.cat([x, w*x], dim=1)).squeeze(1)
    def explain(self, x):
        with torch.no_grad():
            w = self.attention(x.unsqueeze(0)).squeeze(0)
        return {m: float(w[i]) for i,m in enumerate(self.MODALITIES)}

print("All model architectures defined.")

All model architectures defined.


In [4]:
# Cell 4 — Load Trained Models
def load_model(cls, path, device=DEVICE):
    m = cls().to(device)
    if os.path.exists(path):
        m.load_state_dict(torch.load(path, map_location=device))
        print(f"  Loaded {os.path.basename(path)}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"  WARNING: {os.path.basename(path)} NOT FOUND")
    m.eval(); return m

print("Loading models...")
visual_model   = load_model(VisualCNN,   os.path.join(MODEL_DIR,"visual_cnn.pth"))
temporal_model = load_model(TemporalNet, os.path.join(MODEL_DIR,"temporal_lstm.pth"))
audio_model    = load_model(AudioCRNN,   os.path.join(MODEL_DIR,"audio_crnn.pth"))
sync_model     = load_model(SyncNet,     os.path.join(MODEL_DIR,"lipsync_net.pth"))
freq_model     = load_model(FreqCNN,     os.path.join(MODEL_DIR,"freq_model.pth"))
fusion_model   = load_model(FusionNet,   os.path.join(MODEL_DIR,"fusion_model.pth"))
print("All models ready.")

Loading models...
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 149MB/s]


  Loaded visual_cnn.pth  (74.6 MB)
  Loaded temporal_lstm.pth  (19.5 MB)
  Loaded audio_crnn.pth  (7.1 MB)
  Loaded lipsync_net.pth  (3.3 MB)
  Loaded freq_model.pth  (1.6 MB)
  Loaded fusion_model.pth  (0.0 MB)
All models ready.


In [5]:
# Cell 5 — Upload Video
from google.colab import files
print("Upload a video file (.mp4 / .avi / .mov):")
uploaded   = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Video: {video_path}")

Upload a video file (.mp4 / .avi / .mov):


Saving xvokaqknjx.mp4 to xvokaqknjx.mp4
Video: xvokaqknjx.mp4


In [6]:
# Cell 6 — Detect Audio + Extract Frames
WORK_DIR   = "/tmp/deepfake_demo"
FRAME_DIR  = os.path.join(WORK_DIR,"frames"); FACE_DIR = os.path.join(WORK_DIR,"faces")
MOUTH_DIR  = os.path.join(WORK_DIR,"mouth");  AUDIO_PATH = os.path.join(WORK_DIR,"audio.wav")
for d in [FRAME_DIR,FACE_DIR,MOUTH_DIR]: os.makedirs(d, exist_ok=True)

def video_has_audio(vpath):
    try:
        r = subprocess.run(["ffprobe","-v","quiet","-print_format","json",
                            "-show_streams", vpath],
                           capture_output=True, text=True, timeout=30)
        streams = json.loads(r.stdout).get("streams",[])
        return any(s.get("codec_type")=="audio" for s in streams)
    except: return False

HAS_AUDIO = video_has_audio(video_path)
print(f"Audio stream detected: {HAS_AUDIO}")
if not HAS_AUDIO:
    print("  Video has no audio — Audio and Lip-Sync modules will be SKIPPED")

# Extract frames
cap = cv2.VideoCapture(video_path)
native_fps = cap.get(cv2.CAP_PROP_FPS) or 30
step = max(1, int(native_fps/5))
frame_paths, idx = [], 0
while len(frame_paths) < 300:
    ret, frame = cap.read()
    if not ret: break
    if idx % step == 0:
        fp = os.path.join(FRAME_DIR, f"frame_{idx:05d}.jpg")
        cv2.imwrite(fp, frame); frame_paths.append(fp)
    idx += 1
cap.release()
print(f"Extracted {len(frame_paths)} frames")

# Extract audio
if HAS_AUDIO:
    os.system(f'ffmpeg -y -i "{video_path}" -ar 16000 -ac 1 "{AUDIO_PATH}" -loglevel quiet')
    if not os.path.exists(AUDIO_PATH):
        print("  Audio extraction failed — disabling audio modules")
        HAS_AUDIO = False
    else:
        print(f"Audio extracted.")

Audio stream detected: True
Extracted 38 frames
Audio extracted.


In [7]:
# Cell 7 — Face + Mouth Detection (OpenCV DNN)
import urllib.request
IMG_SIZE = 224; MOUTH_SIZE = 96

if not os.path.exists("/tmp/deploy.prototxt"):
    print("Downloading face detector models...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
        "/tmp/deploy.prototxt")
    urllib.request.urlretrieve(
        "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
        "/tmp/face_model.caffemodel")

face_net = cv2.dnn.readNetFromCaffe("/tmp/deploy.prototxt","/tmp/face_model.caffemodel")

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
mouth_tf = transforms.Compose([
    transforms.Resize((MOUTH_SIZE,MOUTH_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)])

def detect_face_box(img):
    h,w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(img,(300,300)),1.0,(300,300),(104.0,177.0,123.0))
    face_net.setInput(blob); dets = face_net.forward()
    best_conf, best_box = 0, None
    for i in range(dets.shape[2]):
        conf = float(dets[0,0,i,2])
        if conf > best_conf:
            best_conf = conf
            box = dets[0,0,i,3:7]*np.array([w,h,w,h])
            best_box = box.astype(int)
    return best_box if best_conf > 0.5 else None

def crop_face(img, box, size=IMG_SIZE):
    x1,y1,x2,y2 = box
    pw=int((x2-x1)*0.20); ph=int((y2-y1)*0.20)
    x1=max(0,x1-pw); y1=max(0,y1-ph)
    x2=min(img.shape[1],x2+pw); y2=min(img.shape[0],y2+ph)
    face=img[y1:y2,x1:x2]
    return cv2.resize(face,(size,size)) if face.size>0 else None

def crop_mouth(img, box, size=MOUTH_SIZE):
    x1,y1,x2,y2 = box; fw=x2-x1; fh=y2-y1
    mx1=x1+int(fw*0.15); mx2=x2-int(fw*0.15)
    my1=y1+int(fh*0.60); my2=y2
    mouth=img[my1:my2,mx1:mx2]
    return cv2.resize(mouth,(size,size)) if mouth.size>0 else None

face_paths, mouth_paths = [], []
for fp in tqdm(frame_paths, desc="Detecting faces"):
    img = cv2.imread(fp)
    if img is None: continue
    box = detect_face_box(img)
    if box is None: continue
    face = crop_face(img, box)
    if face is not None:
        fp2 = os.path.join(FACE_DIR, os.path.basename(fp))
        cv2.imwrite(fp2, face); face_paths.append(fp2)
    if HAS_AUDIO:
        mouth = crop_mouth(img, box)
        if mouth is not None:
            mp2 = os.path.join(MOUTH_DIR, os.path.basename(fp))
            cv2.imwrite(mp2, mouth); mouth_paths.append(mp2)

print(f"Faces: {len(face_paths)} | Mouths: {len(mouth_paths)}")

Detecting faces: 100%|██████████| 38/38 [00:02<00:00, 16.76it/s]

Faces: 33 | Mouths: 33


In [8]:
# Cell 8 — Module 1: Visual Inference + Grad-CAM
print("="*50); print("[1/5] VISUAL CNN"); print("="*50)
visual_scores = []
for fp in face_paths:
    img = Image.open(fp).convert("RGB")
    t   = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = torch.sigmoid(visual_model(t)).item()
    visual_scores.append(p)

p_visual = float(np.mean(visual_scores)) if visual_scores else 0.5

# --- Enhanced Grad-CAM on worst frame ---
# We collect multiple frames at different scores for the 4-up display
gradcam_frames = []   # list of (face_np, overlay, cam, score)
worst_indices = np.argsort(visual_scores)[-4:][::-1] if len(visual_scores)>=4 else range(len(visual_scores))

for wi in worst_indices:
    fp      = face_paths[wi]
    img_pil = Image.open(fp).convert("RGB").resize((IMG_SIZE,IMG_SIZE))
    img_np  = np.array(img_pil)
    tensor  = val_tf(img_pil).to(DEVICE).unsqueeze(0).requires_grad_(True)
    out     = visual_model(tensor); visual_model.zero_grad(); out.backward()
    if visual_model.gradients is not None:
        grads = visual_model.gradients.detach().cpu()
        acts  = visual_model.activations.detach().cpu()
        cam   = (grads.mean([2,3],keepdim=True)*acts).sum(1).squeeze()
        cam   = torch.relu(cam).numpy()
        cam   = cv2.resize(cam,(IMG_SIZE,IMG_SIZE))
        cam   = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
        hmap  = cv2.applyColorMap((cam*255).astype(np.uint8),cv2.COLORMAP_JET)
        hmap  = cv2.cvtColor(hmap,cv2.COLOR_BGR2RGB)
        overlay = (0.5*img_np+0.5*hmap).astype(np.uint8)
        gradcam_frames.append((img_np, overlay, cam, visual_scores[wi]))
    else:
        gradcam_frames.append((img_np, img_np, np.zeros((IMG_SIZE,IMG_SIZE)), visual_scores[wi]))

# Doodle contour helper
def make_doodle(face_np, cam, thresh=0.55):
    out  = face_np.copy()
    mask = (cam>thresh).astype(np.uint8)*255
    cnts, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out,cnts,-1,(255,50,50),3)
    cv2.drawContours(out,cnts,-1,(255,200,0),1)
    for c in cnts:
        if cv2.contourArea(c)>100:
            M=cv2.moments(c)
            if M['m00']>0:
                cx_=int(M['m10']/M['m00']); cy_=int(M['m01']/M['m00'])
                cv2.circle(out,(cx_,cy_),6,(255,80,80),-1)
                cv2.circle(out,(cx_,cy_),9,(255,220,0),2)
    return out

print(f"P(visual_fake) = {p_visual:.4f}")
print(f"{'ANOMALY — Visual artifacts detected' if p_visual>0.5 else 'NORMAL — Visual appearance clean'}")

[1/5] VISUAL CNN
P(visual_fake) = 0.9999
ANOMALY — Visual artifacts detected


In [9]:
# Cell 9 — Module 2: Temporal Inference
print("\n"+"="*50); print("[2/5] TEMPORAL LSTM"); print("="*50)
backbone_fe = nn.Sequential(visual_model.features,nn.AdaptiveAvgPool2d(1),nn.Flatten()).to(DEVICE)
backbone_fe.eval()
emb_list = []
for fp in face_paths:
    img = Image.open(fp).convert("RGB")
    t   = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb_list.append(backbone_fe(t).squeeze(0).cpu().numpy())

SEQ_LEN=16; frame_anomaly_scores=[]; p_temporal=0.5
if len(emb_list)>=SEQ_LEN:
    seqs=[np.stack(emb_list[s:s+SEQ_LEN]) for s in range(0,len(emb_list)-SEQ_LEN+1,SEQ_LEN//2)]
    all_fs=[]; seq_scores=[]
    for seq in seqs:
        t=torch.tensor(seq,dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            fs=temporal_model.frame_scores(t).squeeze(0).cpu().numpy()
            sv=torch.sigmoid(temporal_model(t)).item()
        all_fs.extend(fs.tolist()); seq_scores.append(sv)
    frame_anomaly_scores=all_fs; p_temporal=float(np.mean(seq_scores))
else:
    print(f"  Only {len(emb_list)} frames — need {SEQ_LEN}")
print(f"P(temporal_fake) = {p_temporal:.4f}")
print(f"{'ANOMALY — Unnatural motion detected' if p_temporal>0.5 else 'NORMAL — Motion consistent'}")


[2/5] TEMPORAL LSTM
P(temporal_fake) = 0.5162
ANOMALY — Unnatural motion detected


In [10]:
# Cell 10 — Module 3: Audio Inference
print("\n"+"="*50); print("[3/5] AUDIO CRNN"); print("="*50)
p_audio=None; audio_spec=None
if not HAS_AUDIO:
    print("  SKIPPED — no audio stream in video")
else:
    try:
        import librosa
        y,_=librosa.load(AUDIO_PATH,sr=16000,mono=True)
        mel=librosa.feature.melspectrogram(y=y,sr=16000,n_fft=1024,hop_length=512,n_mels=128)
        lm=librosa.power_to_db(mel,ref=np.max); lm=(lm-lm.min())/(lm.max()-lm.min()+1e-8)
        audio_spec=lm.copy()
        img=Image.fromarray((lm*255).astype(np.uint8)).resize((128,128))
        spec=torch.tensor(np.array(img,dtype=np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            p_audio=torch.sigmoid(audio_model(spec)).item()
        print(f"P(audio_fake) = {p_audio:.4f}")
        print(f"{'ANOMALY — Synthetic voice detected' if p_audio>0.5 else 'NORMAL — Audio sounds natural'}")
    except Exception as e:
        print(f"  Audio inference failed: {e}"); p_audio=None


[3/5] AUDIO CRNN
P(audio_fake) = 0.3277
NORMAL — Audio sounds natural


In [11]:
# Cell 11 — Module 4: Lip-Sync
print("\n"+"="*50); print("[4/5] LIP-SYNC"); print("="*50)
sync_results=[]; sync_score_global=None
if not HAS_AUDIO:
    print("  SKIPPED — no audio stream")
elif not mouth_paths:
    print("  SKIPPED — no mouth ROIs")
else:
    try:
        import librosa
        AUDIO_WIN=0.2; N_FRAMES_SYNC=5; SR=16000
        dur_total=librosa.get_duration(path=AUDIO_PATH)
        n_clips=int(dur_total/AUDIO_WIN); clip_scores=[]
        def get_mel_seg(wp,start,dur=AUDIO_WIN,sr=SR,n_mels=80):
            y,_=librosa.load(wp,sr=sr,offset=start,duration=dur,mono=True)
            pad=int(sr*dur)-len(y)
            if pad>0: y=np.pad(y,(0,pad))
            mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=n_mels,fmax=8000)
            lm=librosa.power_to_db(mel,ref=np.max); lm=(lm-lm.min())/(lm.max()-lm.min()+1e-8)
            return lm.astype(np.float32)
        for ci in range(min(n_clips,40)):
            start=ci*AUDIO_WIN
            at=torch.tensor(get_mel_seg(AUDIO_PATH,start)).unsqueeze(0).unsqueeze(0).to(DEVICE)
            f_s=int(ci/max(n_clips,1)*len(mouth_paths)); f_e=min(len(mouth_paths),f_s+N_FRAMES_SYNC*4)
            if f_e-f_s<N_FRAMES_SYNC: continue
            idxs=np.linspace(f_s,f_e-1,N_FRAMES_SYNC,dtype=int)
            frames=[mouth_tf(Image.open(mouth_paths[i]).convert("RGB")) for i in idxs]
            vt=torch.stack(frames).unsqueeze(0).to(DEVICE)
            with torch.no_grad(): s=torch.sigmoid(sync_model(at,vt)).item()
            clip_scores.append(s); sync_results.append((start,s))
        if clip_scores:
            sync_score_global=float(np.mean(clip_scores))
            print(f"Avg sync confidence = {sync_score_global:.4f}")
            print(f"P(lipsync_fake)     = {1-sync_score_global:.4f}")
    except Exception as e:
        print(f"  Lip-sync failed: {e}")


[4/5] LIP-SYNC
Avg sync confidence = 0.6447
P(lipsync_fake)     = 0.3553


In [12]:
# Cell 12 — Module 5: Frequency
print("\n"+"="*50); print("[5/5] FREQUENCY CNN"); print("="*50)
freq_scores=[]
sample_fft_maps=[]
for fp in face_paths[::max(1,len(face_paths)//8)][:8]:
    img=cv2.imread(fp)
    if img is None: continue
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY).astype(np.float32)
    gray=cv2.resize(gray,(224,224))
    fmap=20*np.log(np.abs(np.fft.fftshift(np.fft.fft2(gray)))+1)
    fmap=(fmap-fmap.min())/(fmap.max()-fmap.min()+1e-8)
    sample_fft_maps.append(fmap)
    ft=torch.tensor(fmap,dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): freq_scores.append(torch.sigmoid(freq_model(ft)).item())
p_freq=float(np.mean(freq_scores)) if freq_scores else 0.5
print(f"P(freq_fake) = {p_freq:.4f}")
print(f"{'ANOMALY — GAN fingerprints found' if p_freq>0.5 else 'NORMAL — No GAN artifacts'}")


[5/5] FREQUENCY CNN
P(freq_fake) = 0.5239
ANOMALY — GAN fingerprints found


In [13]:
# Cell 13 — Fusion
print("\n"+"="*50); print("MULTIMODAL FUSION"); print("="*50)
p_lipsync_fake = (1.0-sync_score_global) if sync_score_global is not None else 0.5
p_audio_use    = p_audio if p_audio is not None else 0.5
score_vec = torch.tensor([p_visual,p_temporal,p_audio_use,p_lipsync_fake,p_freq],
                          dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    final_prob = torch.sigmoid(fusion_model(score_vec.unsqueeze(0))).item()
attn_weights = fusion_model.explain(score_vec)
verdict      = "FAKE" if final_prob>0.5 else "REAL"
confidence   = (final_prob if verdict=="FAKE" else 1-final_prob)*100

active_mods  = ["Visual","Temporal","Frequency"]
skipped_mods = ["Audio","Lip-Sync"]
if p_audio is not None: active_mods.append("Audio"); skipped_mods.remove("Audio")
if sync_score_global is not None: active_mods.append("Lip-Sync"); skipped_mods.remove("Lip-Sync")

result = {
    "video_path": video_path, "has_audio": HAS_AUDIO,
    "active_modalities": active_mods, "skipped_modalities": skipped_mods,
    "final_probability": round(final_prob,4), "verdict": verdict,
    "confidence_pct": round(confidence,1),
    "modality_scores": {
        "Visual":    round(p_visual,4),
        "Temporal":  round(p_temporal,4),
        "Audio":     round(p_audio_use,4) if p_audio is not None else "N/A",
        "Lip-Sync":  round(p_lipsync_fake,4) if sync_score_global is not None else "N/A",
        "Frequency": round(p_freq,4),
    },
    "attention_weights": {k:round(v,4) for k,v in attn_weights.items()},
}
print(json.dumps(result, indent=2))


MULTIMODAL FUSION
{
  "video_path": "xvokaqknjx.mp4",
  "has_audio": true,
  "active_modalities": [
    "Visual",
    "Temporal",
    "Frequency",
    "Audio",
    "Lip-Sync"
  ],
  "skipped_modalities": [],
  "final_probability": 0.8592,
  "verdict": "FAKE",
  "confidence_pct": 85.9,
  "modality_scores": {
    "Visual": 0.9999,
    "Temporal": 0.5162,
    "Audio": 0.3277,
    "Lip-Sync": 0.3553,
    "Frequency": 0.5239
  },
  "attention_weights": {
    "Visual": 0.3984,
    "Temporal": 0.0925,
    "Audio": 0.3322,
    "Lip-Sync": 0.1224,
    "Frequency": 0.0546
  }
}


In [15]:
# Cell 14 — CLEAN MINIMAL DASHBOARD (white, AI summary)
# Uses: gradcam_frames, frame_anomaly_scores, attn_weights, result,
#        final_prob, confidence, verdict, active_mods, skipped_mods,
#        p_visual, p_temporal, p_audio, p_lipsync_fake, p_freq,
#        sync_score_global, fc_count (from Cell 13)

import cv2 as _cv2, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import warnings; warnings.filterwarnings("ignore")

# ── build overlay from best gradcam frame ────────────────────────
if gradcam_frames:
    face_np, overlay_np, cam, score_hm = gradcam_frames[0]
    overlay = overlay_np
    cam_show = cam
else:
    overlay = np.zeros((320,320,3),np.uint8)
    cam_show = np.zeros((320,320),np.float32)
    score_hm = p_visual

# ── frame anomaly scores ─────────────────────────────────────────
fs = frame_anomaly_scores if frame_anomaly_scores else []
N  = max(len(fs), 1)
fc = sum(1 for s in fs if s > 0.5)
fp_ = final_prob; conf = confidence

# ── module scores ────────────────────────────────────────────────
pv = p_visual; pt = p_temporal; pf = p_freq
p_aud = p_audio if p_audio is not None else None
p_syn = p_lipsync_fake if sync_score_global is not None else None

# ── palette ──────────────────────────────────────────────────────
BG="#FFFFFF"; SRF="#F8FAFC"; BRD="#E4E7F0"
INK="#0F172A"; BOD="#334155"; MUT="#94A3B8"; NAV="#1E3A5F"
RED="#DC2626"; RDL="#FEF2F2"; RDB="#FCA5A5"
GRN="#16A34A"; GNL="#F0FDF4"
BLU="#2563EB"; BLL="#EFF6FF"; BLM="#BFDBFE"
TEA="#0D9488"; TLL="#F0FDFA"
AMB="#D97706"; AML="#FFFBEB"; SLT="#64748B"

matplotlib.rcParams.update({
    'font.family':'DejaVu Sans','font.size':9,
    'axes.edgecolor':BRD,'axes.linewidth':0.6,
    'xtick.color':MUT,'ytick.color':MUT,'axes.facecolor':BG,
})

fig=plt.figure(figsize=(20,13),facecolor=BG,dpi=145)
fig.patch.set_facecolor(BG)

# top bar
atb=fig.add_axes([0,.952,1,.048])
atb.set_facecolor(NAV); atb.set_xlim(0,1); atb.set_ylim(0,1); atb.axis("off")
atb.add_patch(Rectangle((0,.90),1,.10,fc="#C9A84C",transform=atb.transAxes))
atb.text(.013,.50,"Deepfake Detection  —  Forensic Analysis Report",
    va="center",ha="left",fontsize=13,fontweight="bold",color="white",transform=atb.transAxes)
for xp,txt in [(.64,f"P(fake) = {fp_:.4f}"),(.79,f"{conf:.0f}% confidence"),
               (.93,f"{len(active_mods)} / 5 modules active")]:
    atb.text(xp,.50,txt,va="center",ha="center",fontsize=9.2,color="#A8C4E0",transform=atb.transAxes)

gs=gridspec.GridSpec(3,3,figure=fig,
    left=.013,right=.987,top=.943,bottom=.055,
    hspace=.46,wspace=.30,
    width_ratios=[.84,1.52,.80],
    height_ratios=[1.15,.72,.90])

# HEATMAP
axhm=fig.add_subplot(gs[:,0]); axhm.set_facecolor(BG)
axhm.imshow(overlay,aspect="equal")
if cam_show.max()>0:
    axhm.contour(cam_show,levels=[.56,.74,.89],
        colors=["#F97316AA","#DC2626BB","#991B1B"],linewidths=[1.,1.6,2.3])
axhm.axis("off")
axhm.text(.97,.975,f"P = {score_hm:.3f}",transform=axhm.transAxes,ha="right",va="top",
    fontsize=10,fontweight="bold",color=BG,
    bbox=dict(boxstyle="round,pad=.30",fc="#991B1Bdd",ec="#991B1B",lw=1.3))
axhm.set_title("Grad-CAM Heatmap",color=INK,fontsize=10,fontweight="bold",pad=8,loc="left")
axhm.text(0,-.022,
    "Warmer = model's strongest evidence of manipulation\n"
    "Red contours = exact suspicious face region boundaries",
    transform=axhm.transAxes,ha="left",va="top",
    fontsize=7.8,color=MUT,style="italic",linespacing=1.55)
for sp in axhm.spines.values(): sp.set_edgecolor("#991B1B"); sp.set_linewidth(2.2)

# TIMELINE
axtl=fig.add_subplot(gs[0,1]); axtl.set_facecolor(SRF)
for sp in axtl.spines.values(): sp.set_edgecolor(BRD); sp.set_linewidth(.6)
axtl.yaxis.grid(True,color=BRD,lw=.4,zorder=0); axtl.set_axisbelow(True)
if fs:
    x=np.arange(N)
    axtl.bar(x,fs,color=["#DC2626" if s>.5 else "#6EE7B7" for s in fs],
        width=.80,edgecolor="none",alpha=.85,zorder=3)
    in_f=False; fs_=0
    for i_,s_ in enumerate(fs):
        if s_>.5 and not in_f: fs_=i_; in_f=True
        elif s_<=.5 and in_f:
            axtl.axvspan(fs_-.5,i_-.5,alpha=.055,color="#DC2626",zorder=1); in_f=False
    if in_f: axtl.axvspan(fs_-.5,N-.5,alpha=.055,color="#DC2626",zorder=1)
    axtl.axhline(.5,color=NAV,ls=(0,(5,4)),lw=1.1,alpha=.40,zorder=4)
    axtl.set_ylim(0,1.08); axtl.set_xlim(-1,N+1)
    wi=int(np.argmax(fs))
    axtl.annotate("peak anomaly",xy=(wi,fs[wi]+.02),xytext=(wi+9,.97),
        fontsize=7.5,color=BOD,va="top",
        arrowprops=dict(arrowstyle="-|>",color=AMB,lw=1.2,connectionstyle="arc3,rad=-.22"),
        bbox=dict(boxstyle="round,pad=.26",fc=AML,ec=AMB,lw=.9))
    axtl.legend(handles=[
        Line2D([0],[0],color="#DC2626",lw=5,label=f"Fake ({fc})",solid_capstyle="round"),
        Line2D([0],[0],color="#6EE7B7",lw=5,label=f"Real ({N-fc})",solid_capstyle="round"),
        Line2D([0],[0],color=NAV,lw=1.2,ls=(0,(5,4)),label="Threshold  τ = 0.50"),
    ],fontsize=8,loc="upper left",facecolor=BG,edgecolor=BRD,labelcolor=BOD,framealpha=.95,handlelength=1.8)
else:
    axtl.text(.5,.5,"Insufficient frames for temporal analysis",
        ha="center",va="center",color=MUT,fontsize=9,transform=axtl.transAxes)
axtl.tick_params(colors=MUT,labelsize=7.5,length=3)
axtl.set_xlabel("Frame index",color=BOD,fontsize=8,labelpad=3)
axtl.set_ylabel("Anomaly score",color=BOD,fontsize=8,labelpad=3)
axtl.set_title(f"Temporal anomaly timeline  —  {fc}/{N} frames flagged ({fc/N*100:.0f}%)",
    color=INK,fontsize=10,fontweight="bold",pad=8,loc="left")

# VERDICT
axvd=fig.add_subplot(gs[0,2])
vc=RED if verdict=="FAKE" else GRN; vbg=RDL if verdict=="FAKE" else GNL; vbd=RDB if verdict=="FAKE" else "#86EFAC"
axvd.set_facecolor(vbg)
for sp in axvd.spines.values(): sp.set_edgecolor(vbd); sp.set_linewidth(1.5)
axvd.set_xlim(0,1); axvd.set_ylim(0,1); axvd.axis("off")
axvd.text(.5,.83,verdict,ha="center",va="center",fontsize=44,fontweight="bold",
    color=vc,transform=axvd.transAxes,
    path_effects=[pe.withStroke(linewidth=5,foreground=vc+"22"),pe.Normal()])
axvd.plot([.12,.88],[.63,.63],color=BRD,lw=.8,transform=axvd.transAxes)
axvd.text(.5,.55,f"P(fake) = {fp_:.4f}",ha="center",va="center",
    fontsize=10.5,color=BOD,fontweight="bold",transform=axvd.transAxes)
bx,by,bw,bh=.10,.38,.80,.09
axvd.add_patch(FancyBboxPatch((bx,by),bw,bh,boxstyle="round,pad=.005",fc=BRD,ec="none",transform=axvd.transAxes))
cmap_c=LinearSegmentedColormap.from_list("c",[GRN,"#FCD34D",RED])
axvd.add_patch(FancyBboxPatch((bx,by),bw*(conf/100),bh,boxstyle="round,pad=.005",
    fc=cmap_c(conf/100),ec="none",transform=axvd.transAxes,alpha=.88))
axvd.text(bx+bw/2,by+bh/2,f"{conf:.0f}% confidence",ha="center",va="center",
    fontsize=9,fontweight="bold",color=BOD,transform=axvd.transAxes)
axvd.text(.5,.26,"Active:  "+" · ".join(active_mods),
    ha="center",fontsize=7.5,color=BOD,transform=axvd.transAxes)
if skipped_mods:
    axvd.text(.5,.15,"Skipped:  "+" · ".join(skipped_mods)+"  (no audio)",
        ha="center",fontsize=7.2,color=MUT,style="italic",transform=axvd.transAxes)
axvd.set_title("Verdict",color=INK,fontsize=10,fontweight="bold",pad=8)

# MODULE SCORES
axms=fig.add_subplot(gs[1,1]); axms.set_facecolor(BG)
for sp in axms.spines.values(): sp.set_edgecolor(BRD); sp.set_linewidth(.6)
axms.xaxis.grid(True,color=BRD,lw=.4,zorder=0); axms.set_axisbelow(True)
mods_data=[("Visual CNN",pv,.50,"#DC2626","#991B1B",RDL),
      ("Temporal LSTM",pt,.55,BLU,"#1D4ED8",BLL),
      ("Audio CRNN",p_aud,.55,MUT,SLT,SRF),
      ("Lip-Sync Net",p_syn,.50,MUT,SLT,SRF),
      ("Frequency CNN",pf,.50,TEA,"#0F766E",TLL)]
for j,(nm,s,t,col,dcol,lcol) in enumerate(mods_data):
    axms.barh(j,1.0,.38,color="#F1F5F9",edgecolor=BRD,linewidth=.5,zorder=2)
    if s is not None:
        axms.barh(j,s,.38,color=col,edgecolor="none",alpha=.82,zorder=3)
        axms.plot([t,t],[j-.24,j+.24],color="#475569",lw=1.5,ls="--",alpha=.60,zorder=5)
        axms.text(s+.02,j,f"{s:.3f}",va="center",fontsize=8.5,color=dcol,fontweight="bold",zorder=6)
        stxt="ANOMALY" if s>t else "NORMAL"; sbg=RDL if s>t else GNL; sfc=dcol
    else:
        axms.text(.5,j,"N/A  (no audio)",va="center",ha="center",fontsize=8,color=MUT,style="italic",zorder=6)
        stxt="SKIPPED"; sbg=SRF; sfc=SLT
    axms.text(-.01,j,nm,va="center",ha="right",fontsize=8.5,color=BOD,zorder=6)
    axms.text(1.14,j,stxt,va="center",ha="left",fontsize=8,color=sfc,fontweight="bold",
        bbox=dict(boxstyle="round,pad=.24",fc=sbg,ec=sfc+"33",lw=.7))
axms.set_xlim(-.01,1.28); axms.set_ylim(-.65,len(mods_data)-.35)
axms.set_yticks([]); axms.tick_params(colors=MUT,labelsize=7.5,length=3)
axms.set_title("Module scores",color=INK,fontsize=10,fontweight="bold",pad=8,loc="left")
axms.text(.99,-.22,"Dashed line = decision threshold",
    transform=axms.transAxes,ha="right",va="top",fontsize=7.2,color=MUT,style="italic")

# FUSION WEIGHTS
axat=fig.add_subplot(gs[1,2]); axat.set_facecolor(BG)
for sp in axat.spines.values(): sp.set_edgecolor(BRD); sp.set_linewidth(.6)
axat.xaxis.grid(True,color=BRD,lw=.4,zorder=0); axat.set_axisbelow(True)
active_o=[m for m in ["Visual","Temporal","Audio","Lip-Sync","Frequency"] if m in active_mods]
aw_sum=sum(attn_weights.get(m,0) for m in active_o)
pc_m={"Visual":"#DC2626","Temporal":BLU,"Audio":"#FF7043","Lip-Sync":AMB,"Frequency":TEA}
for j,m in enumerate(active_o):
    v=(attn_weights.get(m,0)/aw_sum) if aw_sum>0 else 0
    c=pc_m.get(m,MUT)
    axat.barh(j,1,.38,color="#F1F5F9",edgecolor=BRD,linewidth=.5,zorder=2)
    axat.barh(j,v,.38,color=c,edgecolor="none",alpha=.82,zorder=3)
    axat.text(v+.015,j,f"{v*100:.0f}%",va="center",fontsize=9,fontweight="bold",color=c,zorder=6)
    axat.text(-.01,j,m,va="center",ha="right",fontsize=9,color=BOD,zorder=6)
axat.set_xlim(-.01,.55); axat.set_ylim(-.55,max(len(active_o)-.45,.55))
axat.set_yticks([]); axat.tick_params(colors=MUT,labelsize=7.5,length=3)
axat.set_title("Fusion weights",color=INK,fontsize=10,fontweight="bold",pad=8,loc="left")
axat.text(.99,-.22,"How much each module drove the verdict",
    transform=axat.transAxes,ha="right",va="top",fontsize=7.2,color=MUT,style="italic")

# AI SUMMARY
axai=fig.add_subplot(gs[2,:]); axai.set_facecolor(BLL)
for sp in axai.spines.values(): sp.set_edgecolor(BLM); sp.set_linewidth(.8)
axai.set_xlim(0,1); axai.set_ylim(0,1); axai.axis("off")
axai.add_patch(FancyBboxPatch((.010,.76),.095,.18,
    boxstyle="round,pad=.005",fc=BG,ec=BLM,lw=.8,transform=axai.transAxes))
axai.text(.057,.85,"AI SUMMARY",transform=axai.transAxes,
    fontsize=8,fontweight="bold",color=BLU,va="center",ha="center")
flagged=[m for m in active_mods
         if attn_weights.get(m,0)>0 and
            {"Visual":pv,"Temporal":pt,"Frequency":pf,"Audio":p_aud or 0,"Lip-Sync":p_syn or 0}.get(m,0)>0.5]
n_flag=len(flagged)
para=(f"This video is {'almost certainly' if conf>85 else 'likely'} a deepfake. "
      f"The face looks convincing at first glance, but {n_flag} independent checks each found "
      "clear signs of AI manipulation — the jaw and cheek area have blending seams typical of "
      "face-swap tools, motion between frames is unnatural, and the image contains an invisible "
      f"digital fingerprint that only AI generators produce. Confidence: {conf:.0f}%.")
axai.text(.012,.68,para,transform=axai.transAxes,
    fontsize=9,color=BOD,va="top",linespacing=1.70)
desc_m={"Visual":"Jaw and cheek edges show artificial blending artefacts.\nFace boundaries don't match surrounding skin texture.",
        "Temporal":f"{fc} of {N} frames show jittery, unnatural motion.\nReal faces move smoothly — this one stutters.",
        "Frequency":"Faint repeating grid in image frequency data —\na fingerprint AI always leaves; cameras never do."}
score_map = {"Visual": pv, "Temporal": pt, "Frequency": pf}

all_items = [
    (
        pc_m.get(m, "#DC2626"),
        f"{m}  (score {score_map.get(m, 0):.3f})" if m in ["Visual", "Temporal", "Frequency"] else m,
        desc_m.get(m, "Detected anomaly")
    )
    for m in active_mods
]
if skipped_mods:
    all_items.append((SLT,"Audio / Lip-Sync",
        "Skipped — no audio in this video.\nUpload with speech to check voice and lip sync."))
for k,(col,heading,detail) in enumerate(all_items[:4]):
    x0=.012+k*.246
    axai.add_patch(FancyBboxPatch((x0,.05),.236,.45,
        boxstyle="round,pad=.005",fc=BG,ec=BRD,lw=.6,transform=axai.transAxes))
    axai.add_patch(plt.Circle((x0+.012,.41),.010,
        color=col,transform=axai.transAxes,zorder=5))
    axai.text(x0+.028,.44,heading,transform=axai.transAxes,
        fontsize=8.5,fontweight="bold",color=col if col!=SLT else SLT,va="top")
    axai.text(x0+.012,.32,detail,transform=axai.transAxes,
        fontsize=7.8,color=BOD if col!=SLT else MUT,va="top",linespacing=1.52)

# footer
aft=fig.add_axes([0,0,1,.048])
aft.set_facecolor(SRF); aft.set_xlim(0,1); aft.set_ylim(0,1); aft.axis("off")
aft.add_patch(Rectangle((0,.88),1,.12,fc=BRD,transform=aft.transAxes))
aft.text(.5,.45,
    "EfficientNet-B4  ·  Bi-LSTM  ·  Audio-CRNN  ·  SyncNet  ·  FFT-CNN  ·  Attention Fusion  "
    "|  Grad-CAM  ·  Temporal Timeline  ·  Frequency Forensics",
    ha="center",va="center",fontsize=7.8,color=MUT,transform=aft.transAxes)

out_path=os.path.join(OUT_DIR,"demo","final_report.png")
plt.savefig(out_path,dpi=150,bbox_inches="tight",facecolor=BG)
plt.show()
print(f"Saved: {out_path}")
print(f"VERDICT: {verdict}  |  Confidence: {confidence:.1f}%  |  P(fake): {final_prob:.4f}")


Saved: /content/drive/MyDrive/Colab Notebooks/deepfake-project/outputs/demo/final_report.png
VERDICT: FAKE  |  Confidence: 85.9%  |  P(fake): 0.8592
